In [21]:
def print_circuit_infos(circuits):
    for circuit in circuits:
        print(circuit.depth(), circuit.count_ops())

In [22]:
from qiskit_qaoa.utils.hamiltonian_utils import get_Q_and_hamiltonian

# Hamiltonian
filename = 'test_N2_W2'
data_file = f'/lustre/scratch127/qpg/jc59/new_qubo_formulation/oriented/qubo_data/qubo_data_{filename}.gfa.pkl'

Q, hamiltonian, offset, ising_offset = get_Q_and_hamiltonian(data_file)
num_qubits: int = hamiltonian.num_qubits

In [23]:
from qiskit_ibm_runtime import QiskitRuntimeService
# Backend
service = QiskitRuntimeService(name='us_instance')
backend = service.backend(name='ibm_boston')

In [24]:
import numpy as np
from qopt_best_practices.sat_mapping import SATMapper
from qiskit.circuit.library import QAOAAnsatz
from qubo_qaoa.utils.swap_strategy import QUBOSwapStrategy
from qiskit_qaoa.utils.circuit_graph_utils import circuit_to_graph, graph_to_operator

swap_strat = QUBOSwapStrategy.from_line(range(num_qubits))
edge_colouring = {(i, i+1): i % 2 for i in range(num_qubits)}
edge_colouring.update({(i+1, i): i % 2 for i in range(num_qubits)})

qc = QAOAAnsatz(
    cost_operator=hamiltonian,
    reps = 1,
    flatten=True
)
graph = circuit_to_graph(qc, qc.parameters[1])

remapped_g, sat_map, min_sat_layers = SATMapper(timeout=30).remap_graph_with_sat(
    graph=graph, swap_strategy=swap_strat, max_layers = int(num_qubits + np.sqrt(num_qubits) + 61)
)
if remapped_g is None or sat_map is None:
    raise Exception('Failed to find initial layout')

cost_op = graph_to_operator(remapped_g, swap_strat._num_vertices)

In [25]:
from qiskit.circuit import ParameterVector

# Config
p = 1
delta_b = 0.63
delta_g = 0.16
phis = ParameterVector('ϕ', num_qubits)

In [26]:
# Current hardware circuit
from qubo_qaoa.utils.lr_qaoa import get_hardware_LR_qaoa_circuit

hardware_bound_qc, hardware_qc, layout_abstract_qc, layout_cost_circuit = get_hardware_LR_qaoa_circuit(
    p, delta_b, delta_g, num_qubits,
    cost_op, sat_map, backend, edge_colouring, swap_strat,
    None, phis=phis,
)
print_circuit_infos((layout_abstract_qc, hardware_qc))

34 OrderedDict([('cx', 67), ('rz', 38), ('ry', 24), ('measure', 8)])
92 OrderedDict([('rz', 208), ('delay', 173), ('sx', 170), ('cz', 67), ('measure', 8)])


In [27]:
# Current abstract circuit
from qubo_qaoa.utils.lr_qaoa import get_LR_qaoa_circuit
from qiskit import transpile

abstract_bound_qc, abstract_qc, abstract_cost_circuit = get_LR_qaoa_circuit(
    p, delta_b, delta_g, num_qubits,
    hamiltonian, None, phis=phis, measure=True
)
print_circuit_infos((abstract_bound_qc, abstract_qc))

transpile_abstract_qc = transpile(abstract_qc, backend, optimization_level=3)
print_circuit_infos((transpile_abstract_qc,))

11:07:31 - qubo_qaoa.utils.lr_qaoa - INFO - p = 1. Circuit depth: 18


18 OrderedDict([('ry', 24), ('rzz', 22), ('rz', 16), ('measure', 8), ('barrier', 1)])
18 OrderedDict([('ry', 24), ('rzz', 22), ('rz', 16), ('measure', 8), ('barrier', 1)])
202 OrderedDict([('sx', 171), ('rz', 167), ('cz', 79), ('measure', 8), ('x', 2), ('barrier', 1)])


In [28]:
sat_map

{0: 2, 1: 4, 2: 6, 3: 7, 4: 5, 5: 3, 6: 1, 7: 0}

In [29]:
# Circuit construction
from qiskit_qaoa.utils.circuit_graph_utils import circuit_construction
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

singles = cost_op[cost_op.paulis.z.sum(axis=-1) == 1]
doubles = cost_op[cost_op.paulis.z.sum(axis=-1) == 2]
inv_sat_map = {v: k for k, v in sat_map.items()}

# singles = hamiltonian[hamiltonian.paulis.z.sum(axis=-1) == 1]
# doubles = hamiltonian[hamiltonian.paulis.z.sum(axis=-1) == 2]
# inv_sat_map = {i: i for i in range(num_qubits)}


init_state = QuantumCircuit(num_qubits)
for i in range(num_qubits):
    init_state.ry(phis[i], sat_map[i])
    
beta = Parameter("β")
mixer_layer = QuantumCircuit(num_qubits)

indices = {i: swap_strat.inverse_composed_permutation(5).index(sat_map[i]) for i in range(num_qubits)}
for i in range(num_qubits):
    idx = indices[i]
    mixer_layer.ry(-phis[i], idx)
    mixer_layer.rz(-2*beta, idx)
    mixer_layer.ry(phis[i], idx)

circ_dict = circuit_construction(singles, doubles, backend, swap_strat, edge_colouring, {}, p, init_state, mixer_layer)

circuit_construction_abstract, transpile_circuit_construction = (circ_dict['circuit_to_sample'], circ_dict['backend'])
print_circuit_infos((circuit_construction_abstract, transpile_circuit_construction))

34 OrderedDict([('cx', 67), ('rz', 38), ('ry', 24), ('measure', 8)])
92 OrderedDict([('rz', 208), ('delay', 173), ('sx', 170), ('cz', 67), ('measure', 8)])


In [30]:
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as Sampler

simulator = AerSimulator(method='unitary')
# sampler = Sampler.from_backend(simulator)


In [31]:
# abstract_qc.remove_final_measurements()
# abstract_qc.save_unitary()

# circuit_construction_abstract.remove_final_measurements()
# circuit_construction_abstract.save_unitary()

# params = [delta_b, delta_g] + [np.pi/2] * num_qubits
# assigned_abstract_qc = abstract_qc.assign_parameters({p: params[i] for i, p in enumerate(abstract_qc.parameters)})
# assigned_circuit_construction_abstract = circuit_construction_abstract.assign_parameters({p: params[i] for i, p in enumerate(circuit_construction_abstract.parameters)})

# res = simulator.run([assigned_abstract_qc, assigned_circuit_construction_abstract]).result()

In [32]:
# abstract_unitary, construction_unitary = np.asarray(res.results[0].data.unitary), np.asarray(res.results[1].data.unitary)

In [33]:
circ_dict['cost_circuit'].draw(fold=-1)

┌────────────────┐                                                            ┌───┐                                                       ┌───┐                                                       ┌───┐                                  
q_0: ─┤ Rz((-22)*γ[0]) ├───■───────────────────■─────────────────────────────────■──┤ X ├──■──────────────────────────────────■─────────────────┤ X ├──■──────────────────────────────────■─────────────────┤ X ├──■───────────────────────────────
     ┌┴────────────────┴┐┌─┴─┐┌─────────────┐┌─┴─┐                             ┌─┴─┐└─┬─┘┌─┴─┐                    ┌───┐     ┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐                    ┌───┐     ┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐                             
q_1: ┤ Rz((-19.5)*γ[0]) ├┤ X ├┤ Rz(11*γ[0]) ├┤ X ├──■───────────────────────■──┤ X ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├┤ Rz(11*γ[0]) ├──■──┤ X ├──■───────────────────────■──
     ├──────────────────┤└───┘└─────────────┘└───┘┌─┴─┐┌─────────────────┐┌─┴─┐└───┘┌───┐└───┘┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐└───┘└─────────────┘┌───┐└───┘┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐└───┘└─────────────┘┌───┐└───┘┌─┴─┐┌─────────────────┐┌─┴─┐
q_2: ┤ Rz((-19.5)*γ[0]) ├──■───────────────────■──┤ X ├┤ Rz((-2.5)*γ[0]) ├┤ X ├──■──┤ X ├──■──┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├┤ Rz((-2.5)*γ[0]) ├┤ X ├
     ├──────────────────┤┌─┴─┐  ┌──────────┐ ┌─┴─┐└───┘└─────────────────┘└───┘┌─┴─┐└─┬─┘┌─┴─┐└───┘└─────────────┘┌───┐└───┘┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐└───┘└─────────────┘┌───┐└───┘┌─┴─┐  ┌──────────┐ └─┬─┘┌─┴─┐└───┘└─────────────────┘└───┘
q_3: ┤ Rz((-19.5)*γ[0]) ├┤ X ├──┤ Rz(γ[0]) ├─┤ X ├──■───────────────────────■──┤ X ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├──┤ Rz(γ[0]) ├───■──┤ X ├──■───────────────────────■──
     └┬────────────────┬┘└───┘  └──────────┘ └───┘┌─┴─┐    ┌──────────┐   ┌─┴─┐└───┘┌───┐└───┘┌─┴─┐  ┌──────────┐ └─┬─┘┌─┴─┐└───┘└─────────────┘┌───┐└───┘┌─┴─┐  ┌──────────┐ └─┬─┘┌─┴─┐└───┘  └──────────┘ ┌───┐└───┘┌─┴─┐    ┌──────────┐   ┌─┴─┐
q_4: ─┤ Rz((-22)*γ[0]) ├───■───────────────────■──┤ X ├────┤ Rz(γ[0]) ├───┤ X ├──■──┤ X ├──■──┤ X ├──┤ Rz(γ[0]) ├───■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├──┤ Rz(γ[0]) ├───■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├────┤ Rz(γ[0]) ├───┤ X ├
      ├────────────────┤ ┌─┴─┐  ┌──────────┐ ┌─┴─┐└───┘    └──────────┘   └───┘┌─┴─┐└─┬─┘┌─┴─┐└───┘  └──────────┘ ┌───┐└───┘┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐└───┘  └──────────┘ ┌───┐└───┘┌─┴─┐  ┌──────────┐ └─┬─┘┌─┴─┐└───┘    └──────────┘   └───┘
q_5: ─┤ Rz((-22)*γ[0]) ├─┤ X ├──┤ Rz(γ[0]) ├─┤ X ├─────────────────────────────┤ X ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├──┤ Rz(γ[0]) ├───■──┤ X ├─────────────────────────────
      ├────────────────┤ └───┘  └──────────┘ ├───┤                             └───┘     └───┘┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐└───┘└─────────────┘┌───┐└───┘┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐└───┘  └──────────┘      └───┘                             
q_6: ─┤ Rz((-22)*γ[0]) ├───■─────────────────┤ X ├──■─────────────────────────────────────────┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──■───────────────────■────────────────────────────────────
     ┌┴────────────────┴┐┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐                                       └───┘└─────────────┘     └───┘┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐└───┘└─────────────┘     └───┘┌─┴─┐┌─────────────┐┌─┴─┐                                  
q_7: ┤ Rz((-19.5)*γ[0]) ├┤ X ├┤ Rz(11*γ[0]) ├──■──┤ X ├─────────────────────────────────────────────────────────────────────┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──────────────────────────────┤ X ├┤ Rz(11*γ[0]) ├┤ X ├──────────────────────────────────
     └──────────────────┘└───┘└─────────────┘     └───┘                                               

In [34]:
layout_cost_circuit.draw(fold=-1)

┌────────────────┐                                                            ┌───┐                                                       ┌───┐                                                       ┌───┐                                  
q_0: ─┤ Rz((-22)*γ[0]) ├───■───────────────────■─────────────────────────────────■──┤ X ├──■──────────────────────────────────■─────────────────┤ X ├──■──────────────────────────────────■─────────────────┤ X ├──■───────────────────────────────
     ┌┴────────────────┴┐┌─┴─┐┌─────────────┐┌─┴─┐                             ┌─┴─┐└─┬─┘┌─┴─┐                    ┌───┐     ┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐                    ┌───┐     ┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐                             
q_1: ┤ Rz((-19.5)*γ[0]) ├┤ X ├┤ Rz(11*γ[0]) ├┤ X ├──■───────────────────────■──┤ X ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├┤ Rz(11*γ[0]) ├──■──┤ X ├──■───────────────────────■──
     ├──────────────────┤└───┘└─────────────┘└───┘┌─┴─┐┌─────────────────┐┌─┴─┐└───┘┌───┐└───┘┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐└───┘└─────────────┘┌───┐└───┘┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐└───┘└─────────────┘┌───┐└───┘┌─┴─┐┌─────────────────┐┌─┴─┐
q_2: ┤ Rz((-19.5)*γ[0]) ├──■───────────────────■──┤ X ├┤ Rz((-2.5)*γ[0]) ├┤ X ├──■──┤ X ├──■──┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├┤ Rz((-2.5)*γ[0]) ├┤ X ├
     ├──────────────────┤┌─┴─┐  ┌──────────┐ ┌─┴─┐└───┘└─────────────────┘└───┘┌─┴─┐└─┬─┘┌─┴─┐└───┘└─────────────┘┌───┐└───┘┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐└───┘└─────────────┘┌───┐└───┘┌─┴─┐  ┌──────────┐ └─┬─┘┌─┴─┐└───┘└─────────────────┘└───┘
q_3: ┤ Rz((-19.5)*γ[0]) ├┤ X ├──┤ Rz(γ[0]) ├─┤ X ├──■───────────────────────■──┤ X ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├──┤ Rz(γ[0]) ├───■──┤ X ├──■───────────────────────■──
     └┬────────────────┬┘└───┘  └──────────┘ └───┘┌─┴─┐    ┌──────────┐   ┌─┴─┐└───┘┌───┐└───┘┌─┴─┐  ┌──────────┐ └─┬─┘┌─┴─┐└───┘└─────────────┘┌───┐└───┘┌─┴─┐  ┌──────────┐ └─┬─┘┌─┴─┐└───┘  └──────────┘ ┌───┐└───┘┌─┴─┐    ┌──────────┐   ┌─┴─┐
q_4: ─┤ Rz((-22)*γ[0]) ├───■───────────────────■──┤ X ├────┤ Rz(γ[0]) ├───┤ X ├──■──┤ X ├──■──┤ X ├──┤ Rz(γ[0]) ├───■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├──┤ Rz(γ[0]) ├───■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├────┤ Rz(γ[0]) ├───┤ X ├
      ├────────────────┤ ┌─┴─┐  ┌──────────┐ ┌─┴─┐└───┘    └──────────┘   └───┘┌─┴─┐└─┬─┘┌─┴─┐└───┘  └──────────┘ ┌───┐└───┘┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐└───┘  └──────────┘ ┌───┐└───┘┌─┴─┐  ┌──────────┐ └─┬─┘┌─┴─┐└───┘    └──────────┘   └───┘
q_5: ─┤ Rz((-22)*γ[0]) ├─┤ X ├──┤ Rz(γ[0]) ├─┤ X ├─────────────────────────────┤ X ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├──┤ Rz(γ[0]) ├───■──┤ X ├─────────────────────────────
      ├────────────────┤ └───┘  └──────────┘ ├───┤                             └───┘     └───┘┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐└───┘└─────────────┘┌───┐└───┘┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐└───┘  └──────────┘      └───┘                             
q_6: ─┤ Rz((-22)*γ[0]) ├───■─────────────────┤ X ├──■─────────────────────────────────────────┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──■───────────────────■────────────────────────────────────
     ┌┴────────────────┴┐┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐                                       └───┘└─────────────┘     └───┘┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐└───┘└─────────────┘     └───┘┌─┴─┐┌─────────────┐┌─┴─┐                                  
q_7: ┤ Rz((-19.5)*γ[0]) ├┤ X ├┤ Rz(11*γ[0]) ├──■──┤ X ├─────────────────────────────────────────────────────────────────────┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──────────────────────────────┤ X ├┤ Rz(11*γ[0]) ├┤ X ├──────────────────────────────────
     └──────────────────┘└───┘└─────────────┘     └───┘                                               

In [35]:
print_circuit_infos([layout_cost_circuit, circ_dict['cost_circuit']])

29 OrderedDict([('cx', 67), ('rz', 30)])
29 OrderedDict([('cx', 67), ('rz', 30)])


In [36]:
circuit_construction_abstract.draw(fold=-1)

┌──────────┐ ┌────────────────┐                                                            ┌───┐                                                       ┌───┐                                                       ┌───┐               ┌───────────┐   ┌───────────────┐ ┌──────────┐                              ┌─┐                        
q_0: ┤ Ry(ϕ[7]) ├─┤ Rz((-22)*γ[0]) ├───■───────────────────■─────────────────────────────────■──┤ X ├──■──────────────────────────────────■─────────────────┤ X ├──■──────────────────────────────────■─────────────────┤ X ├──────■────────┤ Ry(-ϕ[4]) ├───┤ Rz((-2)*β[0]) ├─┤ Ry(ϕ[4]) ├──────────────────────────────┤M├────────────────────────
     ├──────────┤┌┴────────────────┴┐┌─┴─┐┌─────────────┐┌─┴─┐                             ┌─┴─┐└─┬─┘┌─┴─┐                    ┌───┐     ┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐                    ┌───┐     ┌─┴─┐┌─────────────┐└─┬─┘    ┌─┴─┐      └───────────┘   └───────────────┘ └──────────┘┌───────────┐┌───────────────┐└╥┘┌──────────┐   ┌─┐      
q_1: ┤ Ry(ϕ[6]) ├┤ Rz((-19.5)*γ[0]) ├┤ X ├┤ Rz(11*γ[0]) ├┤ X ├──■───────────────────────■──┤ X ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├┤ Rz(11*γ[0]) ├──■──────┤ X ├────────────■────────────────────────────────■──────┤ Ry(-ϕ[5]) ├┤ Rz((-2)*β[0]) ├─╫─┤ Ry(ϕ[5]) ├───┤M├──────
     ├──────────┤├──────────────────┤└───┘└─────────────┘└───┘┌─┴─┐┌─────────────────┐┌─┴─┐└───┘┌───┐└───┘┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐└───┘└─────────────┘┌───┐└───┘┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐└───┘└─────────────┘┌───┐    └───┘          ┌─┴─┐      ┌─────────────────┐   ┌─┴─┐    ├───────────┤├───────────────┤ ║ ├──────────┤   └╥┘   ┌─┐
q_2: ┤ Ry(ϕ[0]) ├┤ Rz((-19.5)*γ[0]) ├──■───────────────────■──┤ X ├┤ Rz((-2.5)*γ[0]) ├┤ X ├──■──┤ X ├──■──┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──■─────────────────┤ X ├──────■────────────┤ X ├──────┤ Rz((-2.5)*γ[0]) ├───┤ X ├────┤ Ry(-ϕ[3]) ├┤ Rz((-2)*β[0]) ├─╫─┤ Ry(ϕ[3]) ├────╫────┤M├
     ├──────────┤├──────────────────┤┌─┴─┐  ┌──────────┐ ┌─┴─┐└───┘└─────────────────┘└───┘┌─┴─┐└─┬─┘┌─┴─┐└───┘└─────────────┘┌───┐└───┘┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐└───┘└─────────────┘┌───┐└───┘┌─┴─┐  ┌──────────┐ └─┬─┘    ┌─┴─┐          └───┘      └─────────────────┘   └───┘    ├───────────┤├───────────────┤ ║ ├──────────┤┌─┐ ║    └╥┘
q_3: ┤ Ry(ϕ[5]) ├┤ Rz((-19.5)*γ[0]) ├┤ X ├──┤ Rz(γ[0]) ├─┤ X ├──■───────────────────────■──┤ X ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├──┤ Rz(γ[0]) ├───■──────┤ X ├────────────■────────────────────────────────■──────┤ Ry(-ϕ[6]) ├┤ Rz((-2)*β[0]) ├─╫─┤ Ry(ϕ[6]) ├┤M├─╫─────╫─
     ├──────────┤└┬────────────────┬┘└───┘  └──────────┘ └───┘┌─┴─┐    ┌──────────┐   ┌─┴─┐└───┘┌───┐└───┘┌─┴─┐  ┌──────────┐ └─┬─┘┌─┴─┐└───┘└─────────────┘┌───┐└───┘┌─┴─┐  ┌──────────┐ └─┬─┘┌─┴─┐└───┘  └──────────┘ ┌───┐    └───┘          ┌─┴─┐          ┌──────────┐      ┌─┴─┐    ├───────────┤├───────────────┤ ║ ├──────────┤└╥┘ ║ ┌─┐ ║ 
q_4: ┤ Ry(ϕ[1]) ├─┤ Rz((-22)*γ[0]) ├───■───────────────────■──┤ X ├────┤ Rz(γ[0]) ├───┤ X ├──■──┤ X ├──■──┤ X ├──┤ Rz(γ[0]) ├───■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├──┤ Rz(γ[0]) ├───■──┤ X ├──■─────────────────┤ X ├──────■────────────┤ X ├──────────┤ Rz(γ[0]) ├──────┤ X ├────┤ Ry(-ϕ[2]) ├┤ Rz((-2)*β[0]) ├─╫─┤ Ry(ϕ[2]) ├─╫──╫─┤M├─╫─
     ├──────────┤ ├────────────────┤ ┌─┴─┐  ┌──────────┐ ┌─┴─┐└───┘    └──────────┘   └───┘┌─┴─┐└─┬─┘┌─┴─┐└───┘  └──────────┘ ┌───┐└───┘┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐└───┘  └──────────┘ ┌───┐└───┘┌─┴─┐  ┌──────────┐ └─┬─┘    ┌─┴─┐      ┌───┴───┴───┐   ┌──┴──────────┴─┐ ┌──┴───┴───┐└───────────┘└──────┬─┬──────┘ ║ └──────────┘ ║  ║ └╥┘ ║ 
q_5: ┤ Ry(ϕ[4]) ├─┤ Rz((-22)*γ[0]) ├─┤ X ├──┤ Rz(γ[0]) ├─┤ X ├─────────────────────────────┤ X ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├──┤ Rz(γ[0]) ├───■──────┤ X ├──────┤ Ry(-ϕ[7]) ├───┤ Rz((-2)*β[0

In [37]:
layout_abstract_qc.draw(fold=-1)

┌──────────┐ ┌────────────────┐                                                            ┌───┐                                                       ┌───┐                                                       ┌───┐               ┌───────────┐   ┌───────────────┐ ┌──────────┐             ┌─┐                                         
q_0: ┤ Ry(ϕ[7]) ├─┤ Rz((-22)*γ[0]) ├───■───────────────────■─────────────────────────────────■──┤ X ├──■──────────────────────────────────■─────────────────┤ X ├──■──────────────────────────────────■─────────────────┤ X ├──────■────────┤ Ry(-ϕ[4]) ├───┤ Rz((-2)*β[0]) ├─┤ Ry(ϕ[4]) ├─────────────┤M├─────────────────────────────────────────
     ├──────────┤┌┴────────────────┴┐┌─┴─┐┌─────────────┐┌─┴─┐                             ┌─┴─┐└─┬─┘┌─┴─┐                    ┌───┐     ┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐                    ┌───┐     ┌─┴─┐┌─────────────┐└─┬─┘    ┌─┴─┐      └───────────┘   └───────────────┘ └──────────┘┌───────────┐└╥┘┌───────────────┐┌──────────┐      ┌─┐   
q_1: ┤ Ry(ϕ[6]) ├┤ Rz((-19.5)*γ[0]) ├┤ X ├┤ Rz(11*γ[0]) ├┤ X ├──■───────────────────────■──┤ X ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├┤ Rz(11*γ[0]) ├──■──────┤ X ├────────────■────────────────────────────────■──────┤ Ry(-ϕ[5]) ├─╫─┤ Rz((-2)*β[0]) ├┤ Ry(ϕ[5]) ├──────┤M├───
     ├──────────┤├──────────────────┤└───┘└─────────────┘└───┘┌─┴─┐┌─────────────────┐┌─┴─┐└───┘┌───┐└───┘┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐└───┘└─────────────┘┌───┐└───┘┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐└───┘└─────────────┘┌───┐    └───┘          ┌─┴─┐      ┌─────────────────┐   ┌─┴─┐    ├───────────┤ ║ ├───────────────┤├──────────┤   ┌─┐└╥┘   
q_2: ┤ Ry(ϕ[0]) ├┤ Rz((-19.5)*γ[0]) ├──■───────────────────■──┤ X ├┤ Rz((-2.5)*γ[0]) ├┤ X ├──■──┤ X ├──■──┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──■─────────────────┤ X ├──────■────────────┤ X ├──────┤ Rz((-2.5)*γ[0]) ├───┤ X ├────┤ Ry(-ϕ[3]) ├─╫─┤ Rz((-2)*β[0]) ├┤ Ry(ϕ[3]) ├───┤M├─╫────
     ├──────────┤├──────────────────┤┌─┴─┐  ┌──────────┐ ┌─┴─┐└───┘└─────────────────┘└───┘┌─┴─┐└─┬─┘┌─┴─┐└───┘└─────────────┘┌───┐└───┘┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐└───┘└─────────────┘┌───┐└───┘┌─┴─┐  ┌──────────┐ └─┬─┘    ┌─┴─┐          └───┘      └─────────────────┘   └───┘    ├───────────┤ ║ ├───────────────┤├──────────┤   └╥┘ ║ ┌─┐
q_3: ┤ Ry(ϕ[5]) ├┤ Rz((-19.5)*γ[0]) ├┤ X ├──┤ Rz(γ[0]) ├─┤ X ├──■───────────────────────■──┤ X ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├──┤ Rz(γ[0]) ├───■──────┤ X ├────────────■────────────────────────────────■──────┤ Ry(-ϕ[6]) ├─╫─┤ Rz((-2)*β[0]) ├┤ Ry(ϕ[6]) ├────╫──╫─┤M├
     ├──────────┤└┬────────────────┬┘└───┘  └──────────┘ └───┘┌─┴─┐    ┌──────────┐   ┌─┴─┐└───┘┌───┐└───┘┌─┴─┐  ┌──────────┐ └─┬─┘┌─┴─┐└───┘└─────────────┘┌───┐└───┘┌─┴─┐  ┌──────────┐ └─┬─┘┌─┴─┐└───┘  └──────────┘ ┌───┐    └───┘          ┌─┴─┐          ┌──────────┐      ┌─┴─┐    ├───────────┤ ║ ├───────────────┤├──────────┤┌─┐ ║  ║ └╥┘
q_4: ┤ Ry(ϕ[1]) ├─┤ Rz((-22)*γ[0]) ├───■───────────────────■──┤ X ├────┤ Rz(γ[0]) ├───┤ X ├──■──┤ X ├──■──┤ X ├──┤ Rz(γ[0]) ├───■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├──┤ Rz(γ[0]) ├───■──┤ X ├──■─────────────────┤ X ├──────■────────────┤ X ├──────────┤ Rz(γ[0]) ├──────┤ X ├────┤ Ry(-ϕ[2]) ├─╫─┤ Rz((-2)*β[0]) ├┤ Ry(ϕ[2]) ├┤M├─╫──╫──╫─
     ├──────────┤ ├────────────────┤ ┌─┴─┐  ┌──────────┐ ┌─┴─┐└───┘    └──────────┘   └───┘┌─┴─┐└─┬─┘┌─┴─┐└───┘  └──────────┘ ┌───┐└───┘┌─┴─┐┌─────────────┐└─┬─┘┌─┴─┐└───┘  └──────────┘ ┌───┐└───┘┌─┴─┐  ┌──────────┐ └─┬─┘    ┌─┴─┐      ┌───┴───┴───┐   ┌──┴──────────┴─┐ ┌──┴───┴───┐└───────────┘ ║ └──────┬─┬──────┘└──────────┘└╥┘ ║  ║  ║ 
q_5: ┤ Ry(ϕ[4]) ├─┤ Rz((-22)*γ[0]) ├─┤ X ├──┤ Rz(γ[0]) ├─┤ X ├─────────────────────────────┤ X ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├┤ Rz(10*γ[0]) ├──■──┤ X ├──■─────────────────┤ X ├──■──┤ X ├──┤ Rz(γ[0]) ├───■──────┤ X ├──────┤ Ry(-ϕ[7]) ├───┤ Rz((-2)*β[0

In [38]:
hardware_qc.draw(fold=-1)

global phase: 3π/2
           ┌──────────────┐      ┌────┐     ┌──────────────┐   ┌────┐       ┌──────────┐     ┌────────────────┐                   ┌───────────────┐                                                                    ┌───────────────┐                                                                                                                                                         ┌────┐┌───────┐      ┌────┐┌───────┐          ┌───────────────┐                                                                                                                   ┌────┐┌───────┐                              ┌────┐┌───────┐      ┌───────────────┐                                                                                                                                  ┌────┐┌───────┐                                  ┌────┐    ┌─────────┐      ┌───────────────┐   ┌────┐  ┌──────────────┐      ┌────┐        ┌────────┐   ┌───────────────┐      ┌────┐      ┌──────────────┐       ┌────┐          ┌────────┐                                                                                       ┌─┐                                                                                                                                                 
q_0 -> 14 ─┤ Delay(8[dt]) ├──────┤ √X ├─────┤ Rz(π + ϕ[7]) ├───┤ √X ├───────┤ Rz(7π/2) ├─────┤ Rz((-22)*γ[0]) ├──────────────■────┤ Delay(16[dt]) ├───────────────────────────────────────────────────────────────■────┤ Delay(66[dt]) ├───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────■─────┤ √X ├┤ Rz(π) ├──■───┤ √X ├┤ Rz(π) ├────■─────┤ Delay(83[dt]) ├─────────────────────────────────────────────────────────────────────────────────────────────────────────────────■─┤ √X ├┤ Rz(π) ├──────────────────────■───────┤ √X ├┤ Rz(π) ├──■───┤ Delay(83[dt]) ├────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────■─────┤ √X ├┤ Rz(π) ├──────────────────────■───────────┤ √X ├────┤ Rz(π/2) ├──■───┤ Delay(58[dt]) ├───┤ √X ├──┤ Rz(π - ϕ[4]) ├──────┤ √X ├────────┤ Rz(3π) ├───┤ Rz((-2)*β[0]) ├──────┤ √X ├──────┤ Rz(π + ϕ[4]) ├───────┤ √X ├──────────┤ Rz(3π) ├───────────────────────────────────────────────────────────────────────────────────────┤M├─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
           └────┬────┬────┘ ┌────┴────┴────┐└────┬────┬────┘┌──┴────┴──┐┌───┴──────────┴───┐ └─────┬────┬─────┘ ┌───────┐    │    └─────┬────┬────┘    ┌───────┐    ┌─────────────┐   ┌────┐      ┌───────┐       │    └─────┬────┬────┘┌───────┐                                              ┌───────────────┐                                                   ┌────┐     ┌───────┐    │     ├────┤├───────┤  │   ├────┤├───────┤    │     └─────┬────┬────┘ ┌───────┐                                    ┌────┐┌───────┐                  ┌────┐┌───────┐   ┌────┐┌───────┐ │ ├────┤├───────┤┌─────────────┐       │       ├────┤├───────┤  │   └─────┬────┬────┘┌───────┐                                 ┌────┐┌───────┐                  ┌────┐┌───────┐         ┌────┐     ┌───────┐     │     ├────┤├───────┤ ┌─────────────┐      │           ├────┤    └┬───────┬┘  │   └─────┬────┬────┘┌──┴────┴─┐└──────────────┘      └────┘        └────────┘   └───────────────┘┌─────┴────┴────┐ └──────────────┘       └────┘          └────────┘                                     ┌────┐  ┌──────────────┐     ┌────┐     ┌────────┐└╥┘┌───────────────┐      ┌────┐     ┌──────────────┐      ┌────┐         ┌────────┐                                                       ┌─┐      
q_1 -> 15 ──────┤ √X ├──────┤ Rz(π + ϕ[6]) ├─────┤ √X ├─────┤ Rz(7π/2) ├┤ Rz((-19.5)*γ[0]) ├───────┤ √X ├───────┤ Rz(π) ├────■──────────┤ √X ├─────────┤ Rz(π) ├────┤ Rz(11*γ[0]) ├───┤ √X ├──────┤ Rz(π) ├───────■──────────┤ √X ├─────┤ 

In [39]:
transpile_circuit_construction.draw(fold=-1)

global phase: 3π/2
           ┌──────────────┐      ┌────┐     ┌──────────────┐   ┌────┐       ┌──────────┐     ┌────────────────┐                   ┌───────────────┐                                                                    ┌───────────────┐                                                                                                                                                         ┌────┐┌───────┐      ┌────┐┌───────┐          ┌───────────────┐                                                                                                                   ┌────┐┌───────┐                              ┌────┐┌───────┐      ┌───────────────┐                                                                                                                                  ┌────┐┌───────┐                                  ┌────┐    ┌─────────┐      ┌───────────────┐   ┌────┐  ┌──────────────┐      ┌────┐        ┌────────┐   ┌───────────────┐      ┌────┐      ┌──────────────┐       ┌────┐          ┌────────┐                                                                                       ┌─┐                                                                                                                                                 
q_0 -> 14 ─┤ Delay(8[dt]) ├──────┤ √X ├─────┤ Rz(π + ϕ[7]) ├───┤ √X ├───────┤ Rz(7π/2) ├─────┤ Rz((-22)*γ[0]) ├──────────────■────┤ Delay(16[dt]) ├───────────────────────────────────────────────────────────────■────┤ Delay(66[dt]) ├───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────■─────┤ √X ├┤ Rz(π) ├──■───┤ √X ├┤ Rz(π) ├────■─────┤ Delay(83[dt]) ├─────────────────────────────────────────────────────────────────────────────────────────────────────────────────■─┤ √X ├┤ Rz(π) ├──────────────────────■───────┤ √X ├┤ Rz(π) ├──■───┤ Delay(83[dt]) ├────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────■─────┤ √X ├┤ Rz(π) ├──────────────────────■───────────┤ √X ├────┤ Rz(π/2) ├──■───┤ Delay(58[dt]) ├───┤ √X ├──┤ Rz(π - ϕ[4]) ├──────┤ √X ├────────┤ Rz(3π) ├───┤ Rz((-2)*β[0]) ├──────┤ √X ├──────┤ Rz(π + ϕ[4]) ├───────┤ √X ├──────────┤ Rz(3π) ├───────────────────────────────────────────────────────────────────────────────────────┤M├─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
           └────┬────┬────┘ ┌────┴────┴────┐└────┬────┬────┘┌──┴────┴──┐┌───┴──────────┴───┐ └─────┬────┬─────┘ ┌───────┐    │    └─────┬────┬────┘    ┌───────┐    ┌─────────────┐   ┌────┐      ┌───────┐       │    └─────┬────┬────┘┌───────┐                                              ┌───────────────┐                                                   ┌────┐     ┌───────┐    │     ├────┤├───────┤  │   ├────┤├───────┤    │     └─────┬────┬────┘ ┌───────┐                                    ┌────┐┌───────┐                  ┌────┐┌───────┐   ┌────┐┌───────┐ │ ├────┤├───────┤┌─────────────┐       │       ├────┤├───────┤  │   └─────┬────┬────┘┌───────┐                                 ┌────┐┌───────┐                  ┌────┐┌───────┐         ┌────┐     ┌───────┐     │     ├────┤├───────┤ ┌─────────────┐      │           ├────┤    └┬───────┬┘  │   └─────┬────┬────┘┌──┴────┴─┐└──────────────┘      └────┘        └────────┘   └───────────────┘┌─────┴────┴────┐ └──────────────┘       └────┘          └────────┘                                     ┌────┐  ┌──────────────┐     ┌────┐     ┌────────┐└╥┘┌───────────────┐      ┌────┐     ┌──────────────┐      ┌────┐         ┌────────┐                                                       ┌─┐      
q_1 -> 15 ──────┤ √X ├──────┤ Rz(π + ϕ[6]) ├─────┤ √X ├─────┤ Rz(7π/2) ├┤ Rz((-19.5)*γ[0]) ├───────┤ √X ├───────┤ Rz(π) ├────■──────────┤ √X ├─────────┤ Rz(π) ├────┤ Rz(11*γ[0]) ├───┤ √X ├──────┤ Rz(π) ├───────■──────────┤ √X ├─────┤ 

In [40]:
print_circuit_infos([hardware_qc, transpile_circuit_construction])

92 OrderedDict([('rz', 208), ('delay', 173), ('sx', 170), ('cz', 67), ('measure', 8)])
92 OrderedDict([('rz', 208), ('delay', 173), ('sx', 170), ('cz', 67), ('measure', 8)])
